In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = r"D:\DATA\DVP1_lymphomas\lymphomas.csv"

df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'F', 'P', 'S', 'Æ', 'snomed_text', 'T_text', 'M_text', 'F_text', 'P_text', 'S_text', 'Æ_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

print("Full dataset: ", len(df_all))

In [ ]:
from abmil import ABMILInference

slides = df_all["filename"].tolist()
abmil_path = r"D:\DATA\abmil_checkpoints\abmil_hopt_binary.pt"
inference_path = r"D:\DATA\DVP1_lymphomas\inference_hopt_binary.pkl"

inference = ABMILInference(checkpoint_path=abmil_path, zarr_dir=zarr_dir, slides=slides, cache_path=inference_path)
inference.process_slides()

In [ ]:
results_df = inference.results_dataframe()
print(f"Results for {len(results_df)} slides:\n")
print(results_df.head())

In [ ]:
skipped_slides = inference.get_skipped_slides()
print(f"Skipped slides: {len(skipped_slides)}\n")
print(skipped_slides.head())

In [ ]:
# Save to csv
output_file = r"D:\DATA\DVP1_lymphomas\inference_hopt_binary.csv"
results_df.to_csv(output_file, index=False)

output_file = r"D:\DATA\DVP1_lymphomas\inference_hopt_binary_skipped.csv"
skipped_slides.to_csv(output_file, index=False)

In [ ]:
class_dict = {
    'Normal Tissue': 0, 
    'Morphology Not Applicable / Insufficient Tissue': 0, 
    'Cellular Changes / Abnormal Tissue Structure': 1,
    'Traumatic Lesions': 1, 
    'Congenital Malformations': 1,
    'Pregnancy-Related Tissues/Changes': 1,
    'Obstruction / Fluid Retention / Cysts': 1, 
    'Mechanical Changes / Architectural Distortion': 1,
    'Inflammation': 2, 
    'Fibrosis': 1, 
    'Degeneration / Necrosis / Atrophy': 1, 
    'Material Deposits': 1, 
    'Resection Margin Free': 0, 
    'Resection Margin Uncertain': 3,
    'Resection Margin Not Free': 3, 
    'Proliferative/Pre-neoplastic Changes': 3, 
    'Benign Neoplasm': 3, 
    'Uncertain / Borderline Neoplasm': 3, 
    'In Situ Neoplasm': 3, 
    'Malignant Neoplasm': 3,
}

df_all["M_idx"] = df_all["M_category"].apply(lambda lst: [class_dict[x] for x in lst])
df_all['M_idx'] = df_all['M_idx'].apply(lambda x: max(x) if isinstance(x, list) else x)

# 0: Normal
# 1: Other morphologies
# 2: Inflammation
# 3: Neoplastic Changes, Benign/Uncertain/Borderline, In Situ, Malignant Neoplasm 

In [ ]:
from helper_functions import lists2tuples

# Metadata
df_all = lists2tuples(df_all)

# Inference results
result = r"D:\DATA\DVP1_lymphomas\inference_hopt_binary.csv"
results_df= pd.read_csv(result)

In [ ]:
from abmil import ABMILEvaluation

# Initialize evaluator if you have ground truth labels
evaluator = ABMILEvaluation(results_df, metadata_df = df_all, true_label_col="M_idx")

# Match predictions with ground truth
matched_df = evaluator.match_true_labels(slide_id_col="filename", results_path_col="slide_path")
print(f"Matched {len(matched_df)} slides with ground truth")
print(matched_df.head())

In [ ]:
# Compute metrics
metrics = evaluator.compute_metrics()
print(f"\nMacro AUC: {metrics['auc']:.4f}")
print(f"Per-class AUC: {metrics['per_class_aucs']}")

# Confusion matrix and classification report
evaluator.assessment_report()